# UNSW-NB15 Linear Transformer Baseline

This notebook trains and evaluates one binary anomaly-detection baseline using the
kernelized Linear Transformer implementation in `src/transformer.py`.

Scope: PyTorch training only. ONNX, HLS, FPGA, PYNQ, QAFM, LSA, and LAAA are not used.

## 2. Import dependencies

In [1]:
import gc
import json
import os
import random
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.10.12
PyTorch: 1.13.1+cu116
scikit-learn: 1.2.1


## 3. Path and experiment configuration

In [2]:
PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
DATA_DIR = PROJECT_ROOT / "data" / "unsw_nb15_raw"
TRAIN_CSV = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_CSV = DATA_DIR / "UNSW_NB15_testing-set.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "linear_unsw_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cym")

CONFIG = {
    "seed": 42,
    "batch_size": 256,
    "epochs": 20,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "seq_len": 8,
    "d_model": 16,
    "dim_feedforward": 32,
    "num_layers": 1,
    "dropout": 0.1,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_dir": str(DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
}

print(json.dumps(CONFIG, indent=2))

{
  "seed": 42,
  "batch_size": 256,
  "epochs": 20,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "seq_len": 8,
  "d_model": 16,
  "dim_feedforward": 32,
  "num_layers": 1,
  "dropout": 0.1,
  "device": "cpu",
  "data_dir": "/home/cym/prj2/finn/notebooks/icl_thesis-master/data/unsw_nb15_raw",
  "output_dir": "/home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline"
}


## 4. Fix random seeds

In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CONFIG["seed"])
device = torch.device(CONFIG["device"])
print("Device:", device)

Device: cpu


## 5. Verify the fixed UNSW-NB15 files

In [4]:
# The experiment deliberately uses only these two official filenames.
for required_file in (TRAIN_CSV, TEST_CSV):
    if not required_file.is_file():
        raise FileNotFoundError(f"Required dataset file not found: {required_file}")

print("Selected training file:", TRAIN_CSV)
print("Selected testing file:", TEST_CSV)
print("Other files are not selected:")
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path not in (TRAIN_CSV, TEST_CSV):
        print(" -", path.name)

Selected training file: /home/cym/prj2/finn/notebooks/icl_thesis-master/data/unsw_nb15_raw/UNSW_NB15_training-set.csv
Selected testing file: /home/cym/prj2/finn/notebooks/icl_thesis-master/data/unsw_nb15_raw/UNSW_NB15_testing-set.csv
Other files are not selected:
 - README.md
 - hf_Mireu-Lab_test.csv
 - hf_Mireu-Lab_train.csv


## 6. Load the data

In [5]:
train_df = pd.read_csv(TRAIN_CSV, encoding="utf-8-sig")
test_df = pd.read_csv(TEST_CSV, encoding="utf-8-sig")

print("training-set.csv raw samples:", len(train_df))
print("testing-set.csv raw samples:", len(test_df))
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())
print("Training missing values:", int(train_df.isna().sum().sum()))
print("Testing missing values:", int(test_df.isna().sum().sum()))

training-set.csv raw samples: 175341
testing-set.csv raw samples: 82332
Training shape: (175341, 45)
Testing shape: (82332, 45)
Columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']
Training missing values: 0
Testing missing values: 0


## 7. Identify and validate the label

In [6]:
LABEL_PRIORITY = ["label", "Label", "attack_cat", "class", "target"]
label_column = next((name for name in LABEL_PRIORITY if name in train_df.columns), None)
if label_column is None:
    print("Available columns:", train_df.columns.tolist())
    raise ValueError("No supported label column was found")
if label_column != "label":
    raise ValueError(f"This experiment requires the binary 'label' column, found {label_column!r}")
if label_column not in test_df.columns:
    raise ValueError(f"Testing set is missing label column {label_column!r}")

def binary_labels(series):
    numeric = pd.to_numeric(series, errors="raise").astype(np.int64)
    unique = set(numeric.unique().tolist())
    if not unique.issubset({0, 1}):
        raise ValueError(f"Expected binary labels 0/1, found {sorted(unique)}")
    return numeric.to_numpy(dtype=np.int64)


print("Selected label column:", label_column)
print("Full training label distribution:", train_df[label_column].value_counts().sort_index().to_dict())
print("Test label distribution:", test_df[label_column].value_counts().sort_index().to_dict())

Selected label column: label
Full training label distribution: {0: 56000, 1: 119341}
Test label distribution: {0: 37000, 1: 45332}


## 8. Define feature preprocessing

In [7]:
DROPPED_COLUMNS = [name for name in ["id", "label", "attack_cat"] if name in train_df.columns]
feature_columns = [name for name in train_df.columns if name not in DROPPED_COLUMNS]

if set(feature_columns) != set(test_df.columns) - set(DROPPED_COLUMNS):
    raise ValueError("Training and testing feature columns do not match")

feature_view = train_df[feature_columns]
numerical_columns = feature_view.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [name for name in feature_columns if name not in numerical_columns]

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", make_one_hot_encoder()),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

print("Dropped columns:", DROPPED_COLUMNS)
print("Numerical feature count:", len(numerical_columns))
print("Categorical feature count:", len(categorical_columns))
print("Categorical features:", categorical_columns)

Dropped columns: ['id', 'label', 'attack_cat']
Numerical feature count: 39
Categorical feature count: 3
Categorical features: ['proto', 'service', 'state']


## 9. Create train, validation, and test splits; fit preprocessing on train only

In [8]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=CONFIG["seed"],
    stratify=train_df[label_column],
)

y_train = binary_labels(train_split_df[label_column])
y_val = binary_labels(val_split_df[label_column])
y_test = binary_labels(test_df[label_column])

X_train_raw = train_split_df[feature_columns]
X_val_raw = val_split_df[feature_columns]
X_test_raw = test_df[feature_columns]

X_train_flat = np.asarray(preprocessor.fit_transform(X_train_raw), dtype=np.float32)
X_val_flat = np.asarray(preprocessor.transform(X_val_raw), dtype=np.float32)
X_test_flat = np.asarray(preprocessor.transform(X_test_raw), dtype=np.float32)
feature_names = preprocessor.get_feature_names_out().tolist()

feature_dim = X_train_flat.shape[1]
seq_len = CONFIG["seq_len"]
padded_feature_dim = int(np.ceil(feature_dim / seq_len) * seq_len)
padding_features = padded_feature_dim - feature_dim
input_dim = padded_feature_dim // seq_len

def pad_and_reshape(values):
    if padding_features:
        values = np.pad(values, ((0, 0), (0, padding_features)), mode="constant")
    return np.ascontiguousarray(values.reshape(-1, seq_len, input_dim), dtype=np.float32)


X_train = pad_and_reshape(X_train_flat)
X_val = pad_and_reshape(X_val_flat)
X_test = pad_and_reshape(X_test_flat)

print("train split samples:", len(y_train))
print("val split samples:", len(y_val))
print("test samples:", len(y_test))
print("Train label distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Val label distribution:", dict(zip(*np.unique(y_val, return_counts=True))))
print("Test label distribution:", dict(zip(*np.unique(y_test, return_counts=True))))
print("One-hot output feature dimension:", feature_dim)
print("Padded feature dimension:", padded_feature_dim)
print("Padding features:", padding_features)
print("Model input shape:", X_train.shape)

del X_train_flat, X_val_flat, X_test_flat, X_train_raw, X_val_raw, X_test_raw
del train_split_df, val_split_df, train_df, test_df, feature_view
gc.collect()

train split samples: 149039
val split samples: 26302
test samples: 82332
Train label distribution: {0: 47600, 1: 101439}
Val label distribution: {0: 8400, 1: 17902}
Test label distribution: {0: 37000, 1: 45332}
One-hot output feature dimension: 192
Padded feature dimension: 192
Padding features: 0
Model input shape: (149039, 8, 24)


84

## 10. Build Dataset and DataLoader objects

In [9]:
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

loader_options = {
    "batch_size": CONFIG["batch_size"],
    "num_workers": 0,
    "pin_memory": device.type == "cuda",
}
generator = torch.Generator().manual_seed(CONFIG["seed"])
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 583
Validation batches: 103
Test batches: 322


## 11. Build the Linear Transformer baseline

In [10]:
from src.transformer import LinearUNSWAnomalyDetector

MODEL_CONFIG = {
    "input_dim": input_dim,
    "seq_len": seq_len,
    "d_model": CONFIG["d_model"],
    "dim_feedforward": CONFIG["dim_feedforward"],
    "num_layers": CONFIG["num_layers"],
    "dropout": CONFIG["dropout"],
    "num_classes": 2,
}

model = LinearUNSWAnomalyDetector(**MODEL_CONFIG).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

with torch.no_grad():
    shape_check = model(torch.zeros(2, seq_len, input_dim, device=device))
assert tuple(shape_check.shape) == (2, 2)

print(model)
print("Output shape check:", tuple(shape_check.shape))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

LinearUNSWAnomalyDetector(
  (input_projection): Linear(in_features=24, out_features=16, bias=True)
  (layers): ModuleList(
    (0): _UNSWLinearEncoderBlock(
      (attention): _UNSWLinearAttention(
        (query): Linear(in_features=16, out_features=16, bias=False)
        (key): Linear(in_features=16, out_features=16, bias=False)
        (value): Linear(in_features=16, out_features=16, bias=False)
        (output): Linear(in_features=16, out_features=16, bias=True)
      )
      (feedforward): Sequential(
        (0): Linear(in_features=16, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
      )
      (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (output_norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(i

## 12. Training function

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum = 0.0
    sample_count = 0
    for features, targets in loader:
        features = features.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = targets.size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size
    return loss_sum / sample_count

## 13. Validation function

In [12]:
def calculate_metrics(targets, predictions):
    return {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision_score(targets, predictions, zero_division=0),
        "recall": recall_score(targets, predictions, zero_division=0),
        "f1": f1_score(targets, predictions, zero_division=0),
    }


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    all_targets = []
    all_predictions = []

    for features, targets in loader:
        features = features.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(features)
        loss = criterion(logits, targets)
        predictions = logits.argmax(dim=1)

        batch_size = targets.size(0)
        loss_sum += loss.item() * batch_size
        sample_count += batch_size
        all_targets.append(targets.cpu().numpy())
        all_predictions.append(predictions.cpu().numpy())

    targets = np.concatenate(all_targets)
    predictions = np.concatenate(all_predictions)
    metrics = calculate_metrics(targets, predictions)
    metrics["loss"] = loss_sum / sample_count
    return metrics, targets, predictions

## 14. Test function

In [13]:
def test_model(model, loader, criterion, device):
    metrics, targets, predictions = evaluate_model(model, loader, criterion, device)
    matrix = confusion_matrix(targets, predictions, labels=[0, 1])
    report = classification_report(
        targets,
        predictions,
        labels=[0, 1],
        target_names=["normal", "attack"],
        digits=6,
        zero_division=0,
    )
    return metrics, matrix, report, targets, predictions

## 15. Train the model

In [14]:
history = []
best_val_f1 = -1.0
best_epoch = 0
best_model_path = OUTPUT_DIR / "best_model.pt"
training_started = time.perf_counter()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics, _, _ = evaluate_model(model, val_loader, criterion, device)
    epoch_seconds = time.perf_counter() - epoch_started

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "epoch_seconds": epoch_seconds,
    }
    history.append(row)

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "model_config": MODEL_CONFIG,
                "val_metrics": val_metrics,
            },
            best_model_path,
        )

    print(
        f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
        f"train_loss={train_loss:.6f} | val_loss={val_metrics['loss']:.6f} | "
        f"accuracy={val_metrics['accuracy']:.6f} | "
        f"precision={val_metrics['precision']:.6f} | "
        f"recall={val_metrics['recall']:.6f} | f1={val_metrics['f1']:.6f} | "
        f"seconds={epoch_seconds:.2f}"
    )

training_seconds = time.perf_counter() - training_started
print("Best validation epoch:", best_epoch)
print("Best validation F1:", best_val_f1)
print("Total training seconds:", training_seconds)

Epoch 01/20 | train_loss=0.187676 | val_loss=0.152295 | accuracy=0.935062 | precision=0.916298 | recall=0.995531 | f1=0.954273 | seconds=4.45


Epoch 02/20 | train_loss=0.138831 | val_loss=0.132454 | accuracy=0.939548 | precision=0.926301 | recall=0.989945 | f1=0.957066 | seconds=2.50


Epoch 03/20 | train_loss=0.132192 | val_loss=0.125296 | accuracy=0.939929 | precision=0.943436 | recall=0.969892 | f1=0.956481 | seconds=5.39


Epoch 04/20 | train_loss=0.128924 | val_loss=0.122805 | accuracy=0.941335 | precision=0.947285 | recall=0.967657 | f1=0.957363 | seconds=3.11


Epoch 05/20 | train_loss=0.126681 | val_loss=0.121534 | accuracy=0.941145 | precision=0.944692 | recall=0.970339 | f1=0.957344 | seconds=2.55


...

Epoch 16/20 | train_loss=0.118615 | val_loss=0.117881 | accuracy=0.942932 | precision=0.949515 | recall=0.967601 | f1=0.958473 | seconds=5.09


Epoch 17/20 | train_loss=0.118083 | val_loss=0.116071 | accuracy=0.943807 | precision=0.944276 | recall=0.974975 | f1=0.959380 | seconds=5.22


Epoch 18/20 | train_loss=0.118375 | val_loss=0.115276 | accuracy=0.944415 | precision=0.946012 | recall=0.973914 | f1=0.959760 | seconds=5.03


Epoch 19/20 | train_loss=0.118511 | val_loss=0.115500 | accuracy=0.942552 | precision=0.958284 | recall=0.957267 | f1=0.957776 | seconds=2.72


Epoch 20/20 | train_loss=0.117511 | val_loss=0.114319 | accuracy=0.944073 | precision=0.937018 | recall=0.983968 | f1=0.959919 | seconds=2.67
Best validation epoch: 20
Best validation F1: 0.9599193482466417
Total training seconds: 71.61086154600025


## 16. Evaluate the best checkpoint on the fixed test set

In [15]:
try:
    best_checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
except TypeError:
    # PyTorch 1.13 does not expose the weights_only argument.
    best_checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(best_checkpoint["model_state_dict"])

test_metrics, test_matrix, test_report, test_targets, test_predictions = test_model(
    model, test_loader, criterion, device
)

print("Final test metrics")
print(f"Accuracy : {test_metrics['accuracy']:.6f}")
print(f"Precision: {test_metrics['precision']:.6f}")
print(f"Recall   : {test_metrics['recall']:.6f}")
print(f"F1       : {test_metrics['f1']:.6f}")
print(f"Loss     : {test_metrics['loss']:.6f}")
print("Confusion matrix:\n", test_matrix)
print("Classification report:\n", test_report)

Final test metrics
Accuracy : 0.848066
Precision: 0.786189
Recall   : 0.994529
F1       : 0.878171
Loss     : 0.282273
Confusion matrix:
 [[24739 12261]
 [  248 45084]]
Classification report:
               precision    recall  f1-score   support

      normal   0.990075  0.668622  0.798200     37000
      attack   0.786189  0.994529  0.878171     45332

    accuracy                       0.848066     82332
   macro avg   0.888132  0.831575  0.838185     82332
weighted avg   0.877815  0.848066  0.842232     82332



## 17. Save model checkpoints and training results

In [16]:
final_model_path = OUTPUT_DIR / "final_model.pt"
torch.save(
    {
        "epoch": CONFIG["epochs"],
        "model_state_dict": model.state_dict(),
        "model_config": MODEL_CONFIG,
        "note": "Best checkpoint reloaded after training",
    },
    final_model_path,
)

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_log.csv", index=False)

metrics_summary = {
    "best_epoch": best_epoch,
    "best_val_f1": best_val_f1,
    "test_loss": test_metrics["loss"],
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "training_seconds": training_seconds,
}
pd.DataFrame([metrics_summary]).to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
(OUTPUT_DIR / "metrics_summary.json").write_text(
    json.dumps(metrics_summary, indent=2), encoding="utf-8"
)
pd.DataFrame(
    test_matrix,
    index=["actual_normal", "actual_attack"],
    columns=["pred_normal", "pred_attack"],
).to_csv(OUTPUT_DIR / "confusion_matrix.csv")
(OUTPUT_DIR / "classification_report.txt").write_text(test_report, encoding="utf-8")
(OUTPUT_DIR / "model_config.json").write_text(
    json.dumps(MODEL_CONFIG, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(CONFIG, indent=2), encoding="utf-8"
)

print("Saved model and metric files to:", OUTPUT_DIR)

Saved model and metric files to: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline


## 18. Save preprocessing and future conversion/board-test inputs

In [17]:
preprocess_info = {
    "train_csv": str(TRAIN_CSV),
    "test_csv": str(TEST_CSV),
    "label_column": "label",
    "dropped_columns": ["id", "label", "attack_cat"],
    "task": "binary anomaly detection",
    "normal_label": 0,
    "attack_label": 1,
    "train_csv_samples": len(y_train) + len(y_val),
    "test_csv_samples": len(y_test),
    "train_split_samples": len(y_train),
    "val_split_samples": len(y_val),
    "test_samples": len(y_test),
    "split_random_seed": CONFIG["seed"],
    "validation_fraction": 0.15,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "feature_columns": feature_columns,
    "one_hot_feature_dim": feature_dim,
    "seq_len": seq_len,
    "input_dim": input_dim,
    "padding_features": padding_features,
    "padded_feature_dim": padded_feature_dim,
    "model_input_shape": [None, seq_len, input_dim],
    "preprocessing_fit_scope": "training split only",
}

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
(OUTPUT_DIR / "preprocess_info.json").write_text(
    json.dumps(preprocess_info, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "feature_names.json").write_text(
    json.dumps(feature_names, indent=2), encoding="utf-8"
)

sample_count = min(256, len(X_test))
np.savez_compressed(
    OUTPUT_DIR / "conversion_samples.npz",
    inputs=X_test[:sample_count],
    labels=y_test[:sample_count],
    predictions=test_predictions[:sample_count],
)
np.save(OUTPUT_DIR / "sample_inputs.npy", X_test[:sample_count])
np.save(OUTPUT_DIR / "sample_labels.npy", y_test[:sample_count])

readme = f'''# UNSW-NB15 Linear Transformer Baseline

- Data: `{TRAIN_CSV.name}` for train/validation and `{TEST_CSV.name}` for final testing.
- Split: 85% train and 15% validation from the training CSV, stratified by `label`.
- Preprocessing: train-only median imputation plus StandardScaler for numeric features;
  train-only constant imputation plus OneHotEncoder for categorical features.
- Input: `{feature_dim}` encoded features, padded to `{padded_feature_dim}` and reshaped to
  `[batch, {seq_len}, {input_dim}]`.
- Model: one kernelized Linear Transformer encoder block with `d_model={CONFIG['d_model']}`
  and a two-logit classifier.
- Best validation epoch: `{best_epoch}`.
- Test accuracy: `{test_metrics['accuracy']:.6f}`.
- Test precision: `{test_metrics['precision']:.6f}`.
- Test recall: `{test_metrics['recall']:.6f}`.
- Test F1: `{test_metrics['f1']:.6f}`.

The saved preprocessor, model configuration, and conversion samples are the starting point
for a later model-conversion experiment. No conversion or hardware build is performed here.
'''
(OUTPUT_DIR / "README.md").write_text(readme, encoding="utf-8")

print("Saved preprocessing and conversion-support files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f" - {path.name}: {path.stat().st_size} bytes")

Saved preprocessing and conversion-support files:
 - README.md: 914 bytes
 - best_model.pt: 18283 bytes
 - classification_report.txt: 326 bytes
 - confusion_matrix.csv: 75 bytes
 - conversion_samples.npz: 11547 bytes
 - feature_names.json: 5434 bytes
 - final_model.pt: 17985 bytes
 - metrics_summary.csv: 235 bytes
 - metrics_summary.json: 285 bytes
 - model_config.json: 136 bytes
 - preprocess_info.json: 2227 bytes
 - preprocessor.joblib: 7361 bytes
 - run_config.json: 399 bytes
 - sample_inputs.npy: 196736 bytes
 - sample_labels.npy: 2176 bytes
 - training_log.csv: 2804 bytes
